# 08 — Train E4 MultiScaleCNN without augmentation (optional ablation, RQ4)
- **What changes vs E2:** only the training augmentation is switched off — training images get the same deterministic resize + normalization as validation/test (`get_dataloaders(..., augment=False)`). D025.
- **Everything else is E2:** `configs/config.yaml` → `experiments.E4` takes every setting from `experiments.E2` (architecture, optimizer, LR schedule, max 40 epochs, early stopping) and the same seed, so E4 starts from the same initial weights as E2.
- **Data:** the frozen grouped split (D016) through `src/dataset.py`; the validation set is used for checkpoint selection and early stopping.
- **The test set is not loaded in this notebook** (CONSTITUTION C3, D018).
- **Outputs (Drive):** `03_Checkpoints/E4/E4_best.pt`, `E4_last.pt`; `04_Results/experiments/E4/E4_history.csv`, `E4_curves.png`, `E4_val_per_class_metrics.csv`, `E4_val_confusion_matrix.png`, `E4_training_summary.json`.
- **If Colab disconnects:** reconnect and Run all; training resumes from `E4_last.pt`. To start over, rename or delete `03_Checkpoints/E4/` first.
- **Runtime:** GPU (T4). The first epoch of a new session is slow (~16 min of cold Drive reads); later epochs are fast.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/KNV-DSEB/realwaste-classification.git"
REPO_DIR = Path("/content/realwaste-classification")

if not REPO_DIR.exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
# Forget src.* modules imported earlier in this session, so a re-run uses the code just pulled.
for module_name in [m for m in sys.modules if m == "src" or m.startswith("src.")]:
    del sys.modules[module_name]

print(subprocess.run(["git", "-C", str(REPO_DIR), "log", "-1", "--oneline"],
                     capture_output=True, text=True).stdout)

In [ ]:
import json
from datetime import datetime

import matplotlib.pyplot as plt
import torch
from torch import nn

from src.dataset import get_dataloaders, load_class_mapping, load_frozen_split
from src.evaluate import classification_metrics, per_class_metrics, plot_confusion_matrix, predict
from src.models import build_model
from src.train import build_optimizer, build_scheduler, fit, plot_history
from src.utils import count_parameters, environment_info, get_paths, load_config, set_seed

EXPERIMENT_ID = "E4"
cfg = load_config(REPO_DIR / "configs" / "config.yaml")
exp_cfg = {**cfg["experiments"][cfg["experiments"][EXPERIMENT_ID]["based_on"]], **cfg["experiments"][EXPERIMENT_ID]}
assert exp_cfg["augment"] is False
DISPLAY_NAME = exp_cfg["display_name"]
MAX_EPOCHS = cfg["training"]["epochs_complex"]
paths = get_paths(cfg)
CHECKPOINT_DIR = paths.checkpoint_dir / EXPERIMENT_ID
RESULT_DIR = paths.result_dir / "experiments" / EXPERIMENT_ID
RESULT_DIR.mkdir(parents=True, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device.type != "cuda":
    print("WARNING: no GPU. Runtime -> Change runtime type -> T4 GPU, then Run all.")
amp = exp_cfg["amp"] and device.type == "cuda"
env = environment_info()
print(json.dumps(env, indent=2))
print("device:", device, "| AMP:", amp, "| max epochs:", MAX_EPOCHS)

In [ ]:
split_df = load_frozen_split(cfg)  # SHA256-verified frozen grouped split (D016)
class_names = load_class_mapping(cfg)["class_name"].tolist()
num_classes = len(class_names)
train_loader, val_loader = get_dataloaders(cfg, augment=False, split_df=split_df)  # no training augmentation (D025)
print({s: int(n) for s, n in split_df["split"].value_counts().items()})
print("train batches:", len(train_loader), "| val batches:", len(val_loader))

In [ ]:
model_kwargs = {
    "num_classes": num_classes,
    "dropout": exp_cfg["dropout"],
    "stem_width": exp_cfg["stem_width"],
    "widths": exp_cfg["widths"],
}
set_seed(cfg["project"]["seed"])  # identical initial weights on every fresh run
model = build_model(exp_cfg["model_name"], **model_kwargs).to(device)
optimizer = build_optimizer(model.parameters(), exp_cfg)
scheduler = build_scheduler(optimizer, exp_cfg)
criterion = nn.CrossEntropyLoss()  # unweighted (D009)
parameter_count = count_parameters(model)
print(model)
print(f"Trainable parameters: {parameter_count:,}")

In [ ]:
run_info = {
    "experiment_id": EXPERIMENT_ID,
    "run_id": f"{EXPERIMENT_ID}_{datetime.now():%Y%m%d-%H%M%S}",
    "model_name": exp_cfg["model_name"],
    "display_name": DISPLAY_NAME,
    "augment": False,
    "model_kwargs": model_kwargs,
    "class_names": class_names,
    "split_sha256": cfg["split"]["split_sha256"],
    "git_commit": env["git_commit"],
}
result = fit(
    model, train_loader, val_loader,
    optimizer=optimizer, scheduler=scheduler, criterion=criterion, device=device, amp=amp,
    num_classes=num_classes,
    max_epochs=MAX_EPOCHS,
    patience=cfg["training"]["early_stopping_patience"],
    checkpoint_dir=CHECKPOINT_DIR,
    experiment_id=EXPERIMENT_ID,
    history_path=RESULT_DIR / f"{EXPERIMENT_ID}_history.csv",
    run_info=run_info,
)

In [ ]:
history = result.history
fig = plot_history(history, f"{EXPERIMENT_ID} {DISPLAY_NAME} - training curves")
fig.savefig(RESULT_DIR / f"{EXPERIMENT_ID}_curves.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Validation diagnostics of the selected (best) checkpoint. The test set stays untouched.
best = torch.load(result.best_path, map_location=device)
model.load_state_dict(best["model"])
val = predict(model, val_loader, device, criterion, amp)
val_metrics = classification_metrics(val.y_true, val.y_pred, num_classes)
print("Best checkpoint: epoch", best["epoch"])
print("Validation:", {k: round(v, 4) for k, v in val_metrics.items()},
      "| macro-F1 recorded during training:", round(best["val_metrics"]["macro_f1"], 4))

val_per_class = per_class_metrics(val.y_true, val.y_pred, class_names)
val_per_class.to_csv(RESULT_DIR / f"{EXPERIMENT_ID}_val_per_class_metrics.csv", index=False)
print(val_per_class.to_string(index=False))

fig = plot_confusion_matrix(val.y_true, val.y_pred, class_names,
                            f"{EXPERIMENT_ID} {DISPLAY_NAME} - validation confusion matrix")
fig.savefig(RESULT_DIR / f"{EXPERIMENT_ID}_val_confusion_matrix.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Run record (EXPERIMENT_PROTOCOL "Training records"; PROJECT_CONTROL_CENTER "Experiment Runs").
summary = {
    **result.run_info,
    "seed": cfg["project"]["seed"],
    "batch_size": cfg["training"]["batch_size"],
    "optimizer": exp_cfg["optimizer"],
    "learning_rate": exp_cfg["lr"],
    "weight_decay": exp_cfg["weight_decay"],
    "scheduler": exp_cfg["scheduler"],
    "epochs_planned": MAX_EPOCHS,
    "epochs_completed": int(history["epoch"].max()),
    "stopped_early": result.stopped_early,
    "early_stopping": f"{cfg['training']['early_stopping_monitor']}, patience {cfg['training']['early_stopping_patience']}",
    "augmentation": "none (E4 ablation, D025)",
    "class_weighting": cfg["data"]["class_weighting"],
    "amp": amp,
    "best_epoch": result.best_epoch,
    "selection_metric": cfg["training"]["selection_metric"],
    "best_val_macro_f1": result.best_val_macro_f1,
    "best_val_metrics": val_metrics,
    "parameter_count": parameter_count,
    "training_time_minutes": round(float(history["seconds"].sum()) / 60, 1),
    "checkpoint_path": str(result.best_path),
    "environment": env,
    "config_snapshot": cfg,
    "notes": "",
}
with open(RESULT_DIR / f"{EXPERIMENT_ID}_training_summary.json", "w") as f:
    json.dump(summary, f, indent=2)
print(json.dumps({k: v for k, v in summary.items() if k not in ("environment", "config_snapshot", "class_names")}, indent=2))

## Next
1. Save this notebook with outputs to GitHub (File → Save a copy in GitHub) and send the summary printed above, the training curves and the validation confusion matrix.
2. Copy the run into `PROJECT_CONTROL_CENTER.xlsx` → *Experiment Runs* and `experiments/E4_MultiScaleCNN_NoAug_OPTIONAL/README.md`.
3. Do not evaluate on the test set here; that happens once for E1–E3 in the final-evaluation notebook.